In [1]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Introduction to Parallel Programming with MPI<br><br>MPI Derived Datatypes</font></center></h1>

# <font color='blue'>MPI Derived Datatypes</font>

## MPI data types: why?

- Example: Root reads configuration and broadcasts it to all others
```cpp
// root: read configuration from
// file into struct config
MPI_Bcast(&cfg.nx, 1, MPI_INT, …);
MPI_Bcast(&cfg.ny, 1, MPI_INT, …);
MPI_Bcast(&cfg.du, 1, MPI_DOUBLE,…);
MPI_Bcast(&cfg.it, 1, MPI_INT, …);
```
- Want to do something like:
```cpp
MPI_Bcast(  &cfg, 1, <type cfg>, ...);
```
- According to previous lectures, the follow is doable but <font color='red'>not a decent solution</font>.
  Because it is not portable as no data conversion can take place
```cpp
MPI_Bcast(&cfg, sizeof(cfg), MPI_BYTE, ...);
```

<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
MPI is supposed to support parallel computations across <font color='green'>heterogeneous environments</font> and communication in such environments may require data conversions.</span>
</div>

<img src="figs/array-2d-1.svg" alt="array-2d-1.svg" style="float: right; margin-right: 50px;" width="20%">
<!--```{image} figs/array-2d-1.svg
:width: 200px
```-->

- Example: Send column of matrix (noncontiguous in C):
  - Send each element alone?
  - Manually copy elements out into a contiguous buffer and send it?

## Making an MPI data type

Three steps:

- <font color='green'>Construct</font> with
```cpp
MPI_Type_*(...)
```
- <font color='green'>Commit</font> new data type with
```cpp
MPI_Type_commit(MPI_Datatype * nt)
```
- After use, <font color='green'>deallocate</font> the data type with
```cpp
MPI_Type_free(MPI_Datatype * nt)
```

<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
All three steps above are <font color='green'>local</font> and <font color='green'>non-collective calls</font>!</span>
</div>

## A flexible, vector-like type: MPI_Type_vector

```cpp
int MPI_Type_vector(int count, int blocklength, int stride,
    MPI_Datatype oldtype, MPI_Datatype *newtype)
```

<img src="figs/MPI-Type-vector-1.svg" alt="MPI-Type-vector-1.svg" width="90%">
<!--```{image} figs/MPI-Type-vector-1.svg
:width: 1000px
```-->

## Caveat when using a derived type

- <font color='red'>Caution</font>: Concatenating such types in a send operation can lead to unexpected results!
- count argument to send and others must be handled with care:
```cpp
MPI_Send(buf, 2, nt,...)
```
with `nt` (newtype from previous slide)

<img src="figs/MPI-Type-vector-caveat.svg" alt="MPI-Type-vector-caveat.svg" width="80%">
<!--```{image} figs/MPI-Type-vector-caveat.svg
:width: 800px
```-->

## Derives type size and extent

<img src="figs/derived-type-extent.svg" alt="derived-type-extent.svg" style="float: right; margin-right: 50px;" width="12%">
<!--```{image} figs/derived-type-extent.svg
:width: 80px
```-->

- Get the <font color='green'>total size</font> (in bytes) of datatype in a message

```int MPI_Type_size(MPI_Datatype datatype, int *size);```

- Get the lower bound and the extent (span from the first byte to the last byte) of datatype

```int MPI_Type_get_extent(MPI_Datatype datatype, MPI_Aint *lb, MPI_Aint *extent);```

- `MPI_Aint` is the MPI type for memory addresses or offsets
- MPI allows to change the extent of a datatype using
  - `MPI_Type_create_resized`
    - Sizeof
    - `MPI_Get_address`/`MPI_Aint_diff`



## Sending a column of a matrix in C

<img src="figs/column-matrix-C.svg" alt="column-matrix-C.svg" style="float: right; margin-right: 50px;" width="35%">
<!--```{image} figs/column-matrix-C.svg
:width: 400px
```-->

- Row-major data layout in C $\Longrightarrow$ <font color='red'>cannot use plain array</font>

```cpp
double matrix[30];
MPI_Datatype nt;

// count = nrows, blocklength = 1, 
// stride = ncols
MPI_Type_vector(nrows, 1, ncols, MPI_DOUBLE, &nt);
MPI_Type_commit(&nt);

// send column
MPI_Send(&matrix[1], 1, nt, …);

MPI_Type_free(&nt);
```

## A sub-array type: MPI_Type_create_subarray

```cpp
int MPI_Type_create_subarray(int ndims,
                             const int array_of_sizes[],
                             const int array_of_subsizes[],
                             const int array_of_starts[],
                             int order, MPI_Datatype oldtype, MPI_Datatype *newtype);
```
- `ndims`: dimension of the array
- `array_of_sizes`: array with sizes of array (`ndims` entries)
- `array_of_subsizes`: array with sizes of subarray (`ndims` entries)
- `array_of_starts`: start indices of the subarray inside array (`ndims` entries), start at 0 (also in Fortran)
- order
  - <font color='green'>row-major</font>: `MPI_ORDER_C`
  - <font color='green'>column-major</font>: `MPI_ORDER_FORTRAN`

## Example for a sub-array type: "bulk" of a matrix

<img src="figs/derived-type-subarray.svg" alt="derived-type-subarray.svg" style="float: right; margin-right: 50px;" width="30%">
<!--```{image} figs/derived-type-subarray.svg
:width: 300px
```-->

| Variabel | Value |
| :--- | :--- |
| `dims` | 2 |
| `ar_sizes` | {ncols,nrows} |
| `ar_subsizes` | {ncols-2,nrows-2} |
| `ar_starts` | {1,1} |
| `oldtype` | MPI_INT |

<br> <br>

```cpp
MPI_Type_create_subarray(dims, ar_sizes, ar_subsizes, ar_starts, order, oldtype, &nt);
MPI_Type_commit(&nt); 
// use nt...
MPI_Send(&buf[0], 1, nt, ...); // etc.
MPI_Type_free(&nt);
```

## Most flexible type: MPI_Type_create_struct

- Prescribes structured type that can map to `struct` in C and `type` in Fortran: blocks with arbitrary data types and arbitrary displacements
```cpp
int MPI_Type_create_struct(int count, int array_of_blocklengths[],
    const MPI_Aint array_of_displacements[], const MPI_Datatype array_of_types[],
    MPI_Datatype *newtype)
```

<img src="figs/derived-type-struct.svg" alt="derived-type-struct.svg" width="80%">
<!--```{image} figs/derived-type-struct.svg
:width: 800px
```-->

The contents of `array_of_displacements` are either the displacements in <font color='green'>bytes</font> of the block bases or <font color='green'>MPI addresses</font>

## How to obtain and handle addresses?

```cpp
int MPI_Get_address(const void *location, MPI_Aint *address)
```
```cpp
MPI_Aint MPI_Aint_diff(MPI_Aint addr1, MPI_Aint addr2)
```
```cpp
MPI_Aint MPI_Aint_add(MPI_Aint base, MPI_Aint disp)
```
- Example
```cpp
double a[100];
MPI_Aint a1, a2, disp;
MPI_Get_address(&a[0],  &a1);
MPI_Get_address(&a[50], &a2);
disp = MPI_Aint_diff(a2,a1);
```
Result would usually be  `disp` = 400 (50 x 8)

## Derived data types: summary

- A flexible tool to communicate complex data structures in MPI
- Most important calls:
```cpp
MPI_Type_vector         //(second simplest)
MPI_Type_create_subarray
MPI_Type_create_struct  //(most advanced)
MPI_Type_commit/MPI_Type_free
MPI_Get_address, MPI_Aint_add, MPI_Aint_diff
MPI_Type_get_extent, MPI_Type_size
MPI_Type_create_resized
```
- Other useful features: `MPI_Type_contiguous`, `MPI_Type_indexed`, ...
- <font color='green'>Matching rule</font>: send and receive match if specified basic datatypes match one by one, regardless of displacements
  - Correct displacements at receiver side are automatically matched to the corresponding data items

## Quiz:

1. Which one is a use case for MPI derived datatypes?
   <ol style="list-style-type: lower-alpha;">
   <li>Noncontiguous elements of an array</li>
   <li>A set of members in a <code>struct</code> (in C) or <code>type</code> (in Fortran)</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> both a. and b.
   </details> <p></p>
  
2. Which of the bindings `MPI_Type_*` is most flexible? <br>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> <code>MPI_Type_create_struct</code>
   </details> <p></p>

3. The use of MPI derived datatype is <font color='green'>helpful</font> but can be <font color='red'>unsafe</font>.
   <ol style="list-style-type: lower-alpha;">
   <li>Correct</li>
   <li>Incorrect</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> a.
   </details> <p></p>